# Credit Spread Signal Robustness Test

Tests the stability of credit spread signals across different parameter choices:
- MA smoothing periods
- Z-score lookback windows  
- ROC calculation periods

**Goal:** Determine if findings are robust or curve-fitted

In [ ]:
# Install dependencies (Colab already has these, but just in case)
!pip install -q yfinance pandas numpy matplotlib seaborn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

## Download Data

In [ ]:
start_date = '2010-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

print("Downloading IEI, HYG, SPY...")
iei = yf.download('IEI', start=start_date, end=end_date, progress=False)['Adj Close']
hyg = yf.download('HYG', start=start_date, end=end_date, progress=False)['Adj Close']
spy = yf.download('SPY', start=start_date, end=end_date, progress=False)

data = pd.DataFrame({
    'IEI': iei,
    'HYG': hyg,
    'SPY_Close': spy['Adj Close'],
    'SPY_Open': spy['Open']
})

data = data.dropna()
data['Spread'] = data['IEI'] / data['HYG']

# Forward returns
data['Fwd3D'] = data['SPY_Close'].pct_change(3).shift(-3) * 100
data['Fwd5D'] = data['SPY_Close'].pct_change(5).shift(-5) * 100
data['Fwd10D'] = data['SPY_Close'].pct_change(10).shift(-10) * 100

print(f"Data range: {data.index[0].date()} to {data.index[-1].date()}")
print(f"Total days: {len(data)}")

## Define Parameter Space

In [ ]:
SPREAD_MA_PERIODS = [5, 10, 20, 30, 50]
Z_LOOKBACK_PERIODS = [60, 90, 120, 180, 252, 360]
MA_SLOW_PERIODS = [30, 50, 100]
ROC_PERIODS = [1, 2, 3, 5, 10]

print("PARAMETER SPACE")
print("=" * 60)
print(f"Spread MA Periods: {SPREAD_MA_PERIODS}")
print(f"Z-Score Lookbacks: {Z_LOOKBACK_PERIODS}")
print(f"MA Slow Periods: {MA_SLOW_PERIODS}")
print(f"ROC Periods: {ROC_PERIODS}")

## Calculate All Indicators

In [ ]:
print("Calculating spread moving averages...")
for ma_period in SPREAD_MA_PERIODS:
    data[f'SpreadMA{ma_period}'] = data['Spread'].rolling(window=ma_period).mean()

print("Calculating Z-scores across all lookback periods...")
for ma_period in SPREAD_MA_PERIODS:
    for z_period in Z_LOOKBACK_PERIODS:
        mean = data[f'SpreadMA{ma_period}'].rolling(window=z_period).mean()
        std = data[f'SpreadMA{ma_period}'].rolling(window=z_period).std()
        data[f'Z_MA{ma_period}_LB{z_period}'] = (data[f'SpreadMA{ma_period}'] - mean) / std

print("Calculating ROC across all periods...")
for roc_period in ROC_PERIODS:
    data[f'ROC{roc_period}D'] = data['Spread'].pct_change(roc_period) * 100

print("Calculating MA crossovers...")
for slow_period in MA_SLOW_PERIODS:
    data[f'MA10_Above_MA{slow_period}'] = (data['SpreadMA10'] > data[f'SpreadMA{slow_period}']).astype(int)

print(f"\nTotal indicators calculated: {len([c for c in data.columns if c.startswith(('Z_', 'ROC', 'MA10_'))])}")

## Test Signal Robustness

In [ ]:
results = []

# ============================================================================
# TEST 1: Z-Score Zone [0.0 to 0.25]
# ============================================================================

print("[1/4] Testing Z-Score [0.0-0.25] across parameters...")

for ma_period in SPREAD_MA_PERIODS:
    for z_period in Z_LOOKBACK_PERIODS:
        z_col = f'Z_MA{ma_period}_LB{z_period}'
        
        mask = (data[z_col] >= 0.0) & (data[z_col] < 0.25)
        
        if mask.sum() >= 30:
            for fwd in ['Fwd3D', 'Fwd5D', 'Fwd10D']:
                fwd_returns = data.loc[mask, fwd].dropna()
                
                if len(fwd_returns) >= 30:
                    mean_ret = fwd_returns.mean()
                    std_ret = fwd_returns.std()
                    sharpe = (mean_ret / std_ret) if std_ret > 0 else 0
                    win_rate = (fwd_returns > 0).mean() * 100
                    
                    results.append({
                        'Signal': 'ZScore[0-0.25]',
                        'MA_Period': ma_period,
                        'Z_Lookback': z_period,
                        'ROC_Period': None,
                        'Hold': fwd.replace('Fwd', '').replace('D', ''),
                        'Mean_Return': mean_ret,
                        'Sharpe': sharpe,
                        'Win_Rate': win_rate,
                        'N': len(fwd_returns)
                    })

# ============================================================================
# TEST 2: Extreme Wide [1.0-1.5] → Large Tightening
# ============================================================================

print("[2/4] Testing Extreme Wide → Tightening across parameters...")

for ma_period in SPREAD_MA_PERIODS:
    for z_period in Z_LOOKBACK_PERIODS:
        for roc_period in ROC_PERIODS:
            z_col = f'Z_MA{ma_period}_LB{z_period}'
            roc_col = f'ROC{roc_period}D'
            
            mask = (data[z_col] >= 1.0) & (data[z_col] < 1.5) & (data[roc_col] < -0.3)
            
            if mask.sum() >= 10:
                for fwd in ['Fwd3D', 'Fwd5D', 'Fwd10D']:
                    fwd_returns = data.loc[mask, fwd].dropna()
                    
                    if len(fwd_returns) >= 10:
                        mean_ret = fwd_returns.mean()
                        std_ret = fwd_returns.std()
                        sharpe = (mean_ret / std_ret) if std_ret > 0 else 0
                        win_rate = (fwd_returns > 0).mean() * 100
                        
                        results.append({
                            'Signal': 'ExtremeWide→Tight',
                            'MA_Period': ma_period,
                            'Z_Lookback': z_period,
                            'ROC_Period': roc_period,
                            'Hold': fwd.replace('Fwd', '').replace('D', ''),
                            'Mean_Return': mean_ret,
                            'Sharpe': sharpe,
                            'Win_Rate': win_rate,
                            'N': len(fwd_returns)
                        })

# ============================================================================
# TEST 3: Strong ROC Momentum [1.0% to 2.0%]
# ============================================================================

print("[3/4] Testing Strong ROC Momentum across parameters...")

for roc_period in ROC_PERIODS:
    roc_col = f'ROC{roc_period}D'
    
    mask = (data[roc_col] >= 1.0) & (data[roc_col] < 2.0)
    
    if mask.sum() >= 20:
        for fwd in ['Fwd3D', 'Fwd5D', 'Fwd10D']:
            fwd_returns = data.loc[mask, fwd].dropna()
            
            if len(fwd_returns) >= 20:
                mean_ret = fwd_returns.mean()
                std_ret = fwd_returns.std()
                sharpe = (mean_ret / std_ret) if std_ret > 0 else 0
                win_rate = (fwd_returns > 0).mean() * 100
                
                results.append({
                    'Signal': 'StrongROC[1.0-2.0]',
                    'MA_Period': None,
                    'Z_Lookback': None,
                    'ROC_Period': roc_period,
                    'Hold': fwd.replace('Fwd', '').replace('D', ''),
                    'Mean_Return': mean_ret,
                    'Sharpe': sharpe,
                    'Win_Rate': win_rate,
                    'N': len(fwd_returns)
                })

# ============================================================================
# TEST 4: MA Crossover + Tight Spreads
# ============================================================================

print("[4/4] Testing MA Crossover + Tight across parameters...")

for slow_period in MA_SLOW_PERIODS:
    cross_col = f'MA10_Above_MA{slow_period}'
    
    for ma_period in [10]:
        for z_period in Z_LOOKBACK_PERIODS:
            z_col = f'Z_MA{ma_period}_LB{z_period}'
            
            # Fresh bullish cross (within last 5 days) + tight spreads
            data['Fresh_Cross'] = (data[cross_col] == 1) & (data[cross_col].shift(1) == 0)
            data['Cross_Age'] = 0
            for i in range(1, 6):
                data['Cross_Age'] += data['Fresh_Cross'].shift(i)
            
            mask = (data['Cross_Age'] > 0) & (data[z_col] < 0)
            
            if mask.sum() >= 20:
                for fwd in ['Fwd5D', 'Fwd10D']:
                    fwd_returns = data.loc[mask, fwd].dropna()
                    
                    if len(fwd_returns) >= 20:
                        mean_ret = fwd_returns.mean()
                        std_ret = fwd_returns.std()
                        sharpe = (mean_ret / std_ret) if std_ret > 0 else 0
                        win_rate = (fwd_returns > 0).mean() * 100
                        
                        results.append({
                            'Signal': 'MACross+Tight',
                            'MA_Period': slow_period,
                            'Z_Lookback': z_period,
                            'ROC_Period': None,
                            'Hold': fwd.replace('Fwd', '').replace('D', ''),
                            'Mean_Return': mean_ret,
                            'Sharpe': sharpe,
                            'Win_Rate': win_rate,
                            'N': len(fwd_returns)
                        })

df_results = pd.DataFrame(results)
print(f"\nTotal parameter combinations tested: {len(df_results)}")

## Analyze Results by Signal Type

In [ ]:
for signal_name in df_results['Signal'].unique():
    signal_data = df_results[df_results['Signal'] == signal_name]
    
    print("=" * 80)
    print(f"SIGNAL: {signal_name}")
    print("=" * 80)
    
    print(f"\nParameter combinations tested: {len(signal_data)}")
    print(f"\nSharpe Ratio Distribution:")
    print(f"  Mean:   {signal_data['Sharpe'].mean():.2f}")
    print(f"  Median: {signal_data['Sharpe'].median():.2f}")
    print(f"  Std:    {signal_data['Sharpe'].std():.2f}")
    print(f"  Min:    {signal_data['Sharpe'].min():.2f}")
    print(f"  Max:    {signal_data['Sharpe'].max():.2f}")
    
    pct_positive = (signal_data['Sharpe'] > 0).mean() * 100
    pct_good = (signal_data['Sharpe'] > 1.5).mean() * 100
    pct_excellent = (signal_data['Sharpe'] > 3.0).mean() * 100
    
    print(f"\nRobustness Metrics:")
    print(f"  % Positive Sharpe (>0):    {pct_positive:.1f}%")
    print(f"  % Good Sharpe (>1.5):      {pct_good:.1f}%")
    print(f"  % Excellent Sharpe (>3.0): {pct_excellent:.1f}%")
    
    print(f"\nTop 10 Parameter Combinations:")
    top10 = signal_data.nlargest(10, 'Sharpe')[['MA_Period', 'Z_Lookback', 'ROC_Period', 'Hold', 'Mean_Return', 'Sharpe', 'Win_Rate', 'N']]
    print(top10.to_string(index=False))
    
    print(f"\nPerformance by Holding Period:")
    for hold in sorted(signal_data['Hold'].unique()):
        hold_data = signal_data[signal_data['Hold'] == hold]
        print(f"  {hold}D: Mean Sharpe = {hold_data['Sharpe'].mean():.2f}, "
              f"Median = {hold_data['Sharpe'].median():.2f}, "
              f"% >1.5 = {(hold_data['Sharpe'] > 1.5).mean()*100:.1f}%")
    print()

## Robustness Verdict

In [ ]:
print("=" * 80)
print("ROBUSTNESS VERDICT")
print("=" * 80)

for signal_name in df_results['Signal'].unique():
    signal_data = df_results[df_results['Signal'] == signal_name]
    
    mean_sharpe = signal_data['Sharpe'].mean()
    median_sharpe = signal_data['Sharpe'].median()
    pct_good = (signal_data['Sharpe'] > 1.5).mean() * 100
    std_sharpe = signal_data['Sharpe'].std()
    
    print(f"\n{signal_name}:")
    print(f"  Mean Sharpe: {mean_sharpe:.2f}")
    print(f"  Median Sharpe: {median_sharpe:.2f}")
    print(f"  Std Dev: {std_sharpe:.2f}")
    print(f"  % Good (>1.5): {pct_good:.1f}%")
    
    if median_sharpe > 2.0 and pct_good > 60:
        verdict = "✅ HIGHLY ROBUST - Works across most parameters"
    elif median_sharpe > 1.0 and pct_good > 40:
        verdict = "✅ ROBUST - Works with proper parameter selection"
    elif median_sharpe > 0.5 and pct_good > 20:
        verdict = "⚠️ MODERATELY ROBUST - Requires careful parameter tuning"
    else:
        verdict = "❌ NOT ROBUST - Likely curve-fitted"
    
    print(f"  Verdict: {verdict}")

## Visualizations

In [ ]:
# Sharpe distribution by signal
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Sharpe Ratio Distribution by Signal Type', fontsize=16, fontweight='bold')

for idx, signal_name in enumerate(df_results['Signal'].unique()):
    ax = axes[idx // 2, idx % 2]
    signal_data = df_results[df_results['Signal'] == signal_name]
    
    ax.hist(signal_data['Sharpe'], bins=30, alpha=0.7, edgecolor='black')
    ax.axvline(signal_data['Sharpe'].median(), color='red', linestyle='--', linewidth=2, label=f"Median: {signal_data['Sharpe'].median():.2f}")
    ax.axvline(1.5, color='green', linestyle='--', linewidth=2, label='Good Threshold (1.5)')
    ax.set_title(signal_name, fontweight='bold')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Parameter sensitivity heatmaps
for signal_name in ['ZScore[0-0.25]', 'ExtremeWide→Tight']:
    signal_data = df_results[df_results['Signal'] == signal_name]
    
    if 'Z_Lookback' in signal_data.columns and signal_data['Z_Lookback'].notna().any():
        # Create pivot table
        pivot = signal_data.pivot_table(
            values='Sharpe',
            index='MA_Period',
            columns='Z_Lookback',
            aggfunc='mean'
        )
        
        plt.figure(figsize=(12, 6))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=1.5, vmin=0, vmax=5)
        plt.title(f'{signal_name}: Sharpe by MA Period vs Z-Score Lookback', fontweight='bold')
        plt.xlabel('Z-Score Lookback Period')
        plt.ylabel('MA Smoothing Period')
        plt.tight_layout()
        plt.show()

## Export Results

In [ ]:
# Export to CSV
df_results.to_csv('credit_spread_robustness_results.csv', index=False)
print("✅ Results exported to: credit_spread_robustness_results.csv")

# Summary statistics
summary = df_results.groupby('Signal').agg({
    'Sharpe': ['count', 'mean', 'median', 'std', 'min', 'max'],
    'Mean_Return': 'mean',
    'Win_Rate': 'mean'
}).round(2)

summary.to_csv('credit_spread_robustness_summary.csv')
print("✅ Summary exported to: credit_spread_robustness_summary.csv")

print("\n" + "=" * 80)
print("ROBUSTNESS TEST COMPLETE")
print("=" * 80)